## GeoSpatial data



In [8]:
import duckdb


In [43]:
# Connect to DuckDB and load the spatial extension
con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;") ;


In [35]:
# Define the remote URLs
roads_url = "https://www2.census.gov/geo/tiger/TIGER2024/ROADS/tl_2024_35006_roads.zip"
states_url = "https://www2.census.gov/geo/tiger/TIGER2024/STATE/tl_2024_us_state.zip"

roads_url, states_url

('https://www2.census.gov/geo/tiger/TIGER2024/ROADS/tl_2024_35006_roads.zip',
 'https://www2.census.gov/geo/tiger/TIGER2024/STATE/tl_2024_us_state.zip')

In [44]:
# Extract the base file name
roads_base = roads_url.split("/")[-1].split(".")[0]
states_base = states_url.split("/")[-1].split(".")[0]

roads_base, states_base

('tl_2024_35006_roads', 'tl_2024_us_state')

In [45]:
# Construct GDAL virtual paths for remote zipped shapefiles
# /vsicurl/ streams over HTTP, and /vsizip/ opens the archive on-the-fly
roads_path = f"/vsizip//vsicurl/{roads_url}/{roads_base}.shp"
states_path = f"/vsizip//vsicurl/{states_url}/{states_base}.shp"

roads_path, states_path


('/vsizip//vsicurl/https://www2.census.gov/geo/tiger/TIGER2024/ROADS/tl_2024_35006_roads.zip/tl_2024_35006_roads.shp',
 '/vsizip//vsicurl/https://www2.census.gov/geo/tiger/TIGER2024/STATE/tl_2024_us_state.zip/tl_2024_us_state.shp')

In [39]:
# Export Roads shapefile to GeoParquet
con.execute(f"""
    COPY (
        SELECT * FROM st_read('{roads_path}')
    ) TO '{roads_base}.parquet' (FORMAT PARQUET, COMPRESSION ZSTD);
""")

# Export States shapefile to GeoParquet
con.execute(f"""
    COPY (
        SELECT * FROM st_read('{states_path}')
    ) TO '{states_base}.parquet' (FORMAT PARQUET, COMPRESSION ZSTD);
""")

!ls -la *.parquet


-rw-r--r-- 1 root root  2317411 Nov 18 05:56 roads_35006.parquet
-rw-r--r-- 1 root root  2317411 Nov 18 05:57 tl_2024_35006_roads.parquet
-rw-r--r-- 1 root root 11340375 Nov 18 05:57 tl_2024_us_state.parquet
-rw-r--r-- 1 root root 11340375 Nov 18 05:56 us_states.parquet


In [40]:
duckdb.sql( f'''
  select *
  from './{states_base}.parquet'
  limit 10
  ;
''').df()


,REGION,DIVISION,STATEFP,STATENS,GEOID,GEOIDFQ,STUSPS,NAME,LSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geom
0,3,5,54,01779805,54,0400000US54,WV,West Virginia,00,G4000,A,62266513826,488918898,+38.6472854,-080.6183274,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 170, 137, 0, 0, 20..."
1,3,5,12,00294478,12,0400000US12,FL,Florida,00,G4000,A,138965379385,45968913048,+28.3989775,-082.5143005,"[1, 6, 0, 0, 0, 2, 0, 0, 0, 1, 3, 0, 0, 0, 1, ..."
2,2,3,17,01779784,17,0400000US17,IL,Illinois,00,G4000,A,143778206717,6216848695,+40.1028754,-089.1526108,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 49, 77, 0, 0, 9, 2..."
3,2,4,27,00662849,27,0400000US27,MN,Minnesota,00,G4000,A,206244791203,18937236061,+46.3159573,-094.1996043,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 101, 157, 0, 0, 18..."
4,3,5,24,01714934,24,0400000US24,MD,Maryland,00,G4000,A,25151223822,6979843236,+38.9466584,-076.6744939,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 237, 101, 0, 0, 75..."
5,1,1,44,01219835,44,0400000US44,RI,Rhode Island,00,G4000,A,2677768885,1323681453,+41.5964850,-071.5264901,"[1, 6, 0, 0, 0, 2, 0, 0, 0, 1, 3, 0, 0, 0, 1, ..."
6,4,8,16,01779783,16,0400000US16,ID,Idaho,00,G4000,A,214050504522,2390996667,+44.3484222,-114.5588538,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 48, 137, 0, 0, 26,..."
7,1,1,33,01779794,33,0400000US33,NH,New Hampshire,00,G4000,A,23190211616,1025871482,+43.6727945,-071.5841886,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 224, 64, 0, 0, 214..."
8,3,5,37,01027616,37,0400000US37,NC,North Carolina,00,G4000,A,125935965771,13453455061,+35.5397100,-079.1308636,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 133, 125, 0, 0, 18..."
9,1,1,50,01779802,50,0400000US50,VT,Vermont,00,G4000,A,23872594714,1030642813,+44.0589536,-072.6710173,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 87, 47, 0, 0, 6, 2..."


In [41]:
query = f"""
    SELECT
        roads.* EXCLUDE (geom),
        states.STATEFP,
        states.STUSPS,
        states.NAME AS state_name,
        roads.geom AS road_geometry,
        states.geom AS state_geometry
    FROM read_parquet('{roads_base}.parquet') roads
    LEFT JOIN read_parquet('{states_base}.parquet') states
        ON ST_Within(roads.geom, states.geom)
    WHERE roads.FULLNAME LIKE '%I- 25%'
       OR (roads.RTTYP = 'I'
       -- AND roads.FULLNAME LIKE '%25%'
       )
"""
print(query)



    SELECT 
        roads.* EXCLUDE (geom), 
        states.STATEFP,
        states.STUSPS, 
        states.NAME AS state_name,
        roads.geom AS road_geometry,
        states.geom AS state_geometry
    FROM read_parquet('tl_2024_35006_roads.parquet') roads
    LEFT JOIN read_parquet('tl_2024_us_state.parquet') states
        ON ST_Within(roads.geom, states.geom)
    WHERE roads.FULLNAME LIKE '%I- 25%' 
       OR (roads.RTTYP = 'I' 
       -- AND roads.FULLNAME LIKE '%25%'
       )



In [42]:
gdf_combined = con.sql(query).df()
gdf_combined


,LINEARID,FULLNAME,RTTYP,MTFCC,STATEFP,STUSPS,state_name,road_geometry,state_geometry
0,1104492838447,I- 40,I,S1100,35,NM,New Mexico,"[1, 4, 0, 0, 0, 0, 0, 0, 52, 251, 215, 194, 68...","[2, 4, 0, 0, 0, 0, 0, 0, 211, 25, 218, 194, 67..."
1,1104492838321,I- 40,I,S1100,35,NM,New Mexico,"[1, 4, 0, 0, 0, 0, 0, 0, 4, 251, 215, 194, 129...","[2, 4, 0, 0, 0, 0, 0, 0, 211, 25, 218, 194, 67..."
